# FAKE NEWS DETECTION

This project Involves building a model that will predict with text(news statement) if a news is FAKE or REAL.

This is a SUPERVISED learning with a CLASSIFICATION (BINARY) problem 

#### PROJECT FLOW

        DATA COLLECTION
              ↓
        DATA UNDERSTANDING
              ↓
        DATA CLEANING  
              ↓
        EDA / DATA ANALYSIS
              ↓
        TEXT PREPROCESSING
              ↓
        TRAIN / TEST SPLIT
              ↓
        TEXT VECTORIZATION / ENCODING
              ↓
        MODEL TRAINING & VALIDATION
              ↓
        MODEL EVALUATION
              ↓
        MODEL TUNING
              ↓
        FINAL MODEL
              ↓
        DEPLOYMENT

                    TF-IDF
                       ↓
              ┌────────┼────────┐
              ↓        ↓        ↓
        Logistic     Linear    Naive
        Regression     SVM      Bayes
              ↓        ↓        ↓
              └────────┼────────┘
                       ↓
                  Compare metrics
                       ↓
                 Select final model

In [1]:
import pandas as pd

In [2]:
# Import the data

data=pd.read_excel(r"C:\Users\USER\Downloads\Kaggle DataSet\fake_or_real_news.csv.xlsx")
data.head(5)

,id,title,text,label,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,Unnamed: 135,Unnamed: 136,Unnamed: 137,Unnamed: 138,Unnamed: 139,Unnamed: 140
0,8476,You Can Smell Hillaryâ€™s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10142,Bernie supporters on Twitter erupt in anger ag...,"â€” Kaydee King (@KaydeeKing) November 9, 2016...",FAKE,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7795 entries, 0 to 7794
Columns: 141 entries, id to Unnamed: 140
dtypes: object(141)
memory usage: 8.4+ MB


In [4]:
data.isna().sum()

id               247
title            610
text             866
label           1040
Unnamed: 4      7477
                ... 
Unnamed: 136    7794
Unnamed: 137    7794
Unnamed: 138    7794
Unnamed: 139    7794
Unnamed: 140    7794
Length: 141, dtype: int64

In [5]:
data.columns

Index(['id', 'title', 'text', 'label', 'Unnamed: 4', 'Unnamed: 5',
       'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9',
       ...
       'Unnamed: 131', 'Unnamed: 132', 'Unnamed: 133', 'Unnamed: 134',
       'Unnamed: 135', 'Unnamed: 136', 'Unnamed: 137', 'Unnamed: 138',
       'Unnamed: 139', 'Unnamed: 140'],
      dtype='object', length=141)

### Data Cleaning

In [6]:
# Remove rredundant columns 
df= data[['id', 'title', 'text', 'label']]
df.head()

,id,title,text,label
0,8476,You Can Smell Hillaryâ€™s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"â€” Kaydee King (@KaydeeKing) November 9, 2016...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [7]:
# Check for duplicates
df.duplicated().sum()

np.int64(228)

This can means that 228 rows are exact duplicates of an earlier row across all columns.

The next step is to investigate which rows they are and what is being duplicated.
So let's investigate further

In [8]:
# Display the duplicated rows
duplicates = df[df.duplicated(keep=False)]

duplicates

,id,title,text,label
258,NaN,NaN,NaN,NaN
260,NaN,NaN,NaN,NaN
262,NaN,NaN,NaN,NaN
264,NaN,NaN,NaN,NaN
266,NaN,NaN,NaN,NaN
...,...,...,...,...
7248,NaN,NaN,NaN,NaN
7250,NaN,NaN,NaN,NaN
7252,NaN,NaN,NaN,NaN
7254,NaN,NaN,NaN,NaN


That means these aren't necessarily 233 duplicated news articles. They are likely completely empty rows that appear multiple times.
Pandas consider empty space as duplicated cells

In [9]:
# Count completely empty rows

empty_rows = df[df.isna().all(axis=1)]

print("Completely empty rows:", len(empty_rows))

Completely empty rows: 221


In [10]:
# Check for exact duplicates of non-empty rows
df_clean = df.dropna(how='all')

df_clean.duplicated().sum()

np.int64(8)

Let's check missing data and handle it first, then wiull come back to reconfirm the number duplicate we have, if any.

In [11]:
# Examine for missing data
df.isna().sum().sum()

np.int64(2763)

In [12]:
print(df.isna().sum())
print(f"Total rows with missing values: {df.isna().any(axis=1).sum()}")

id        247
title     610
text      866
label    1040
dtype: int64
Total rows with missing values: 1061


we have 2763 missing values but only 1061 affected rows;
This means the missing values are overlapping in the same records (they are concentrated in the same rows)
We have 13% of the data missing. Best practice is to drop them and keep roughly 7,000 clean rows.

In [13]:
# Drop the missing values

# Create a df copy
df_clean = df.copy()



In [14]:
# Remove only completely empty data
df_clean = df_clean.dropna(how='all')

In [15]:
# Examine the cleaned data

print("Rows remaining:", len(df_clean))
print("Completely empty rows:", df_clean.isna().all(axis=1).sum())

Rows remaining: 7574
Completely empty rows: 0


In [16]:
df_clean.isna().sum().sum()

np.int64(1879)

In [17]:
# Remove rows with missing rows

df_clean = df_clean.dropna(subset=['label'])

print("Rows remaining:", len(df_clean))
print("Missing labels:", df_clean['label'].isna().sum())

Rows remaining: 6755
Missing labels: 0


In [18]:
df_clean.isna().sum().sum()

np.int64(21)

In [19]:
print("Missing text:", df_clean['text'].isna().sum())
print("Usable text:", df_clean['text'].notna().sum())

Missing text: 1
Usable text: 6754


In [20]:
# Examine the number missing brows in text and title columns
print("Both title and text missing:",
      (df_clean['title'].isna() & df_clean['text'].isna()).sum())

print("Title missing, text available:",
      (df_clean['title'].isna() & df_clean['text'].notna()).sum())

print("Text missing, title available:",
      (df_clean['text'].isna() & df_clean['title'].notna()).sum())

Both title and text missing: 0
Title missing, text available: 0
Text missing, title available: 1


In [21]:
# Remove the missing row in text column
df_clean = df_clean.dropna(subset=['text'])

print("Rows remaining:", len(df_clean))
print("Missing text:", df_clean['text'].isna().sum())

Rows remaining: 6754
Missing text: 0


In [22]:
df_clean.isna().sum().sum()

np.int64(20)

This missing rows is coming from ID column and its not needed in the data modelling

The step-by-step approach lets us make a decision based on the role of each column

In [23]:
# Check label values

#print(df_clean['label'].value_counts(dropna=False))
#print("\nUnique labels:")
#print(df_clean['label'].unique())

In [24]:
# instigate the labels with the actual data and others 
print("FAKE:", (df_clean['label'] == 'FAKE').sum())
print("REAL:", (df_clean['label'] == 'REAL').sum())

print("Other labels:",
      (~df_clean['label'].isin(['FAKE', 'REAL'])).sum())

FAKE: 3154
REAL: 3161
Other labels: 439


In [25]:
# Create a new df of labels that are not the correct labels and displays the first 20 rows of those problematic records.

bad_labels = df_clean[
    ~df_clean['label'].isin(['FAKE', 'REAL'])
]

print(bad_labels[['id', 'title', 'text', 'label']].head(5).to_string())

                                                                                     id                                                                                                                                                     title                                                                                                                                                                            text                                                                                                                                                                                                                label
257                       g campaign â€¦ the epic dark saga will unfold this way: A man                                                                                                                           filled with fear and insecurity                                                                                                   created a hatemongering chara

In [26]:
valid_labels = ['FAKE', 'REAL']

print("Total rows:", len(df_clean))

print("Valid labels:",
      df_clean['label'].isin(valid_labels).sum())

print("Invalid labels:",
      (~df_clean['label'].isin(valid_labels)).sum())

Total rows: 6754
Valid labels: 6315
Invalid labels: 439


So this 439 other rows are the corrupted records we identified.

In [27]:
print(df_clean[df_clean['label'].isin(['FAKE', 'REAL'])]['label'].value_counts())

label
REAL    3161
FAKE    3154
Name: count, dtype: int64


In [28]:
# Remove the corrupted labels

valid_labels = ['FAKE', 'REAL']

df_clean = df_clean[
    df_clean['label'].isin(valid_labels)
].copy()

In [29]:
# Verify the label column now

print("Rows remaining:", len(df_clean))
print("\nLabel distribution:")
print(df_clean['label'].value_counts())

print("\nMissing values:")
print(df_clean.isna().sum())

Rows remaining: 6315

Label distribution:
label
REAL    3161
FAKE    3154
Name: count, dtype: int64

Missing values:
id       0
title    0
text     0
label    0
dtype: int64


In [30]:
# Examine the column and values datatypes

print(df_clean.dtypes)

print("\nText column types:")
print(df_clean['text'].map(type).value_counts())

print("\nTitle column types:")
print(df_clean['title'].map(type).value_counts())

print("\nLabel column types:")
print(df_clean['label'].map(type).value_counts())

id       object
title    object
text     object
label    object
dtype: object

Text column types:
text
<class 'str'>                  6314
<class 'datetime.datetime'>       1
Name: count, dtype: int64

Title column types:
title
<class 'str'>    6315
Name: count, dtype: int64

Label column types:
label
<class 'str'>    6315
Name: count, dtype: int64


In [31]:
# I dentify the rwo with the datatime type

non_string_text = df_clean[
    ~df_clean['text'].map(lambda x: isinstance(x, str))
]

print(non_string_text[['id', 'title', 'text', 'label']].to_string())

        id                                  title                 text label
3156  8751  2009 FLASHBACK: â€œWhat Ifâ€ Remixed  2016-08-11 00:00:00  FAKE


In [32]:
# delete the row where datetime is text

df_clean = df_clean[
    df_clean['text'].map(lambda x: isinstance(x, str))
].copy()

This output just revealed something critical in the data: 3156  8751  2009 FLASHBACK: â€œWhat Ifâ€ Remixed  2016-08-11 00:00:00  FAKE.
This means that some values in the text and title columns has character encoding issues which needs to be solved else then model might not learn corrected but with noises that exist in the data\

In [33]:
df_clean.head()

,id,title,text,label
0,8476,You Can Smell Hillaryâ€™s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"â€” Kaydee King (@KaydeeKing) November 9, 2016...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [34]:
# lets install ftfy, a library specifically designed to fix Unicode/encoding problems:

!pip install ftfy

In [35]:
# Import 'fix_text' and apply it in the 'text' and 'title' columns
from ftfy import fix_text

df_clean['title'] = df_clean['title'].apply(fix_text)
df_clean['text'] = df_clean['text'].apply(fix_text)

In [36]:
df_clean.head()

,id,title,text,label
0,8476,You Can Smell Hillary's Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [37]:
patterns = ['â€œ', 'â€', 'â€™', 'â€“', 'â€”']

for pattern in patterns:
    count_title = df_clean['title'].str.contains(pattern, na=False).sum()
    count_text = df_clean['text'].str.contains(pattern, na=False).sum()

    print(f"{pattern} -> titles: {count_title}, texts: {count_text}")

â€œ -> titles: 0, texts: 0
â€ -> titles: 0, texts: 3
â€™ -> titles: 0, texts: 0
â€“ -> titles: 0, texts: 0
â€” -> titles: 0, texts: 0


Fix the 3 text values that still contain corrupted sequence.

In [38]:
df_clean['text'] = df_clean['text'].str.replace(
    'â€', '"', regex=False
)

In [39]:
df_clean['text'] = df_clean['text'].str.replace(
    'â€Ś', '…', regex=False
)

In [40]:
# Confirm
patterns = ['â€œ', 'â€', 'â€™', 'â€“', 'â€”', 'â€Ś']

for pattern in patterns:
    count_title = df_clean['title'].str.contains(pattern, na=False).sum()
    count_text = df_clean['text'].str.contains(pattern, na=False).sum()

    print(f"{pattern} -> titles: {count_title}, texts: {count_text}")

â€œ -> titles: 0, texts: 0
â€ -> titles: 0, texts: 0
â€™ -> titles: 0, texts: 0
â€“ -> titles: 0, texts: 0
â€” -> titles: 0, texts: 0
â€Ś -> titles: 0, texts: 0


In [41]:
# Check for duplicates

print("Exact duplicate rows:", df_clean.duplicated().sum())

print("Duplicate texts:", df_clean['text'].duplicated().sum())

duplicate_texts = df_clean[
    df_clean['text'].duplicated(keep=False)
].sort_values('text')

print(duplicate_texts[['id', 'title', 'label']].head(30).to_string(index=False))

Exact duplicate rows: 0
Duplicate texts: 275
   id                                                                                                       title label
10386                          Israeli Minister sparks Scandal after showing Satisfaction for Italy's Earthquakes  FAKE
10008                           Israeli Minister sparks Scandal after showing Satisfaction at Italy's Earthquakes  FAKE
 8548                                                     A Mormon Reader Says Most Mormons Will Still Back Trump  FAKE
 8626                                                                                                World War 3?  FAKE
 8332                                        MARKETWATCH LEFTIST: MSM's "Blatant" Anti Trump Bias May Be Suicidal  FAKE
 9314                            Southern Poverty Law Center Targets Anti-Jihad Crusaders (Ann Corcoran Honored!)  FAKE
 8601                                   Pro-sovereignty Legislators Demand That Administration End Border Anarchy  

In [42]:
df.duplicated().sum()

np.int64(228)

In [43]:
# Inspect whether the repeated articles have consistent labels.
label_conflicts = (
    df_clean.groupby('text')['label']
    .nunique()
)

print(
    "Duplicate texts with conflicting labels:",
    (label_conflicts > 1).sum()
)

Duplicate texts with conflicting labels: 0


In [44]:
df_clean = df_clean.drop_duplicates(subset='text', keep='first').copy()

print("Rows remaining:", len(df_clean))
print("\nLabel distribution:")
print(df_clean['label'].value_counts())

Rows remaining: 6039

Label distribution:
label
FAKE    3060
REAL    2979
Name: count, dtype: int64


### EDA (EXPLORATORY DATA ANALYSIS)

In [45]:
# Examine the final dataset
print("Dataset shape:", df_clean.shape)
print("\nColumns:")
print(df_clean.columns.tolist())

print("\nData types:")
print(df_clean.dtypes)

print("\nMissing values:")
print(df_clean.isna().sum())

print("\nLabel distribution:")
print(df_clean['label'].value_counts())

Dataset shape: (6039, 4)

Columns:
['id', 'title', 'text', 'label']

Data types:
id       object
title    object
text     object
label    object
dtype: object

Missing values:
id       0
title    0
text     0
label    0
dtype: int64

Label distribution:
label
FAKE    3060
REAL    2979
Name: count, dtype: int64


In [46]:
# Check the text length

df_clean['text_length'] = df_clean['text'].str.len()

print(df_clean['text_length'].describe())

count     6039.000000
mean      4706.938566
std       4083.023784
min          1.000000
25%       1943.500000
50%       3812.000000
75%       6276.500000
max      32518.000000
Name: text_length, dtype: float64


In [47]:
# Compare text length by label to see whether FAKE and REAL articles have noticeably different text-length distributions.
print(
    df_clean.groupby('label')['text_length']
    .describe()
)

        count         mean          std   min     25%     50%     75%      max
label                                                                         
FAKE   3060.0  3978.528758  4215.528607   1.0  1313.5  2610.0  5076.5  32518.0
REAL   2979.0  5455.154079  3800.259152  33.0  3045.0  4829.0  6946.5  32509.0


### Text Processing

This involves preparing the text so that it is suitable for the model.

In [48]:
# create one column containing both the headline and article body (text & title).

df_clean['model_text'] = (
    df_clean['title'].fillna('') + ' ' +
    df_clean['text'].fillna('')
)

df_clean.head(3)

,id,title,text,label,text_length,model_text
0,8476,You Can Smell Hillary's Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE,7518,You Can Smell Hillary's Fear Daniel Greenfield...
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE,2646,Watch The Exact Moment Paul Ryan Committed Pol...
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL,2543,Kerry to go to Paris in gesture of sympathy U....


In [49]:
# Process the text

import re

def preprocess_text(text):
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

In [50]:
# Apply the processed text to the model text column

df_clean['processed_text'] = df_clean['model_text'].apply(preprocess_text)

In [51]:
# view the dataframe 
df_clean.head(3)

,id,title,text,label,text_length,model_text,processed_text
0,8476,You Can Smell Hillary's Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE,7518,You Can Smell Hillary's Fear Daniel Greenfield...,you can smell hillary's fear daniel greenfield...
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE,2646,Watch The Exact Moment Paul Ryan Committed Pol...,watch the exact moment paul ryan committed pol...
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL,2543,Kerry to go to Paris in gesture of sympathy U....,kerry to go to paris in gesture of sympathy u....


In [52]:
# compare the processed_text and combined model_text
print("Before:")
print(df_clean['model_text'].iloc[0])

print("\nAfter:")
print(df_clean['processed_text'].iloc[0])

Before:
You Can Smell Hillary's Fear Daniel Greenfield, a Shillman Journalism Fellow at the Freedom Center, is a New York writer focusing on radical Islam. 
In the final stretch of the election, Hillary Rodham Clinton has gone to war with the FBI. 
The word "unprecedented" has been thrown around so often this election that it ought to be retired. But it's still unprecedented for the nominee of a major political party to go war with the FBI. 
But that's exactly what Hillary and her people have done. Coma patients just waking up now and watching an hour of CNN from their hospital beds would assume that FBI Director James Comey is Hillary's opponent in this election. 
The FBI is under attack by everyone from Obama to CNN. Hillary's people have circulated a letter attacking Comey. There are currently more media hit pieces lambasting him than targeting Trump. It wouldn't be too surprising if the Clintons or their allies were to start running attack ads against the FBI. 
The FBI's leadership

### Model Building



#### Train / Test split

In [53]:
# Train / Test split

from sklearn.model_selection import train_test_split

X = df_clean['processed_text']
y = df_clean['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

Training samples: 4831
Testing samples: 1208

Training labels:
label
FAKE    2448
REAL    2383
Name: count, dtype: int64

Testing labels:
label
FAKE    612
REAL    596
Name: count, dtype: int64


### Feature extraction/vectorization 

The model cannot directly understand:
"the government announced a new policy..."

We need to convert the text into numerical features and we'll use TF-IDF (Term Frequency–Inverse Document Frequency) to achieve.

In [54]:
# Import TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer

In [55]:
# Create the Vectorizer

tfidf = TfidfVectorizer(
    max_features=50000, # use at most 50,000 features
    ngram_range=(1, 2), # Use individual words and two-words combinations
    min_df=2, # ignore words appearing in only one document
    max_df=0.95 # ignore words appearing in more than 95% of documents
)

In [56]:
# Fit the vectorizer on the training data(Text)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [57]:
# Print the result

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (4831, 50000)
Testing TF-IDF shape: (1208, 50000)


#### Model Training -- Logistic Regression

This model will ask: Based on these words/features, what is the probability that this article belongs to each class?

In [58]:
# Let's train a baseline model using LogisticRegression

from sklearn.linear_model import LogisticRegression

model_log = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_log.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [59]:
# make prediction with the trained Model

y_pred = model_log.predict(X_test_tfidf)

print(y_pred[:20])

['FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL'
 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL' 'FAKE' 'FAKE']


#### Model Evaluation

In [60]:
# Import the evaluation mertic libraries

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


In [61]:
# Evaluate the model using the imported libraries
# pos_label tells machine learning to focu on 'FAKE' when calculating precision, recall, and F1.
# So we're simply defining it as the class of interest for this particular evaluation.

log_accuracy = accuracy_score(y_test, y_pred)
log_precision = precision_score(y_test, y_pred, pos_label='FAKE')
log_recall = recall_score(y_test, y_pred, pos_label='FAKE')
log_f1 = f1_score(y_test, y_pred, pos_label='FAKE')

print("Accuracy :", round(log_accuracy, 2))
print("Precision:", round(log_precision, 2))
print("Recall   :", round(log_recall, 2))
print("F1 Score :", round(log_f1, 2))

Accuracy : 0.91
Precision: 0.89
Recall   : 0.94
F1 Score : 0.91


In [62]:
# Get the Classification report

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

        FAKE       0.89      0.94      0.91       612
        REAL       0.93      0.88      0.91       596

    accuracy                           0.91      1208
   macro avg       0.91      0.91      0.91      1208
weighted avg       0.91      0.91      0.91      1208



In [63]:
# Evaluate the confusion matrix

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:
[[573  39]
 [ 70 526]]


#### The Logistics Regression Model Performance:
The Logistic Regression model achieved 91% accuracy on the test set, with an F1-score of 0.91 for both FAKE and REAL classes. 
The model achieved 94% recall for FAKE news, correctly identifying 573 of the 612 fake articles in the test set.


### 2nd Model Training -- LINEAR SVM

we're specifically using a Linear SVM because our input is TF-IDF text data.

The basic idea is:
Linear SVM will try to find the best boundary that separates FAKE articles from REAL articles.

In [64]:
# import LinearSVM
from sklearn.svm import LinearSVC

In [65]:
#Instantiate and train the model

svm_model = LinearSVC(
    random_state=42,
    max_iter = 2000
)

svm_model.fit(X_train_tfidf, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,42


In [66]:
# Make predictions

y_pred_svm= svm_model.predict(X_test_tfidf)
print(y_pred_svm[:20])

['FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL'
 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL' 'FAKE' 'FAKE']


In [67]:
# Evaluate the models performance

svm_accuracy= accuracy_score(y_test, y_pred_svm)

svm_precision = precision_score(
    y_test, y_pred_svm, pos_label='FAKE'
)

svm_recall = recall_score(
    y_test, y_pred_svm, pos_label='FAKE'
)

svm_f1 = f1_score(
    y_test, y_pred_svm, pos_label='FAKE'
)

print("Linear SVM Evaluation")
print("---------------------")
print(f"Accuracy : {svm_accuracy:.2f}")
print(f"Precision: {svm_precision:.2f}")
print(f"Recall   : {svm_recall:.2f}")
print(f"F1 Score : {svm_f1:.2f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

Linear SVM Evaluation
---------------------
Accuracy : 0.93
Precision: 0.92
Recall   : 0.95
F1 Score : 0.94

Classification Report:
              precision    recall  f1-score   support

        FAKE       0.92      0.95      0.94       612
        REAL       0.95      0.92      0.93       596

    accuracy                           0.93      1208
   macro avg       0.93      0.93      0.93      1208
weighted avg       0.93      0.93      0.93      1208


Confusion Matrix:
[[582  30]
 [ 50 546]]


#### Linear SVM Model performance:

The Linear Support Vector Machine (SVM) model achieved **93% accuracy** on the test dataset. 

It recorded a precision of **92%**, recall of **95%**, and F1-score of **0.94** for the FAKE class. For the REAL class, it achieved 95% precision, 92% recall, and an F1-score of 0.93. The overall macro F1-score was 0.93, indicating balanced performance across both classes. 

These results show that the model performed well on the test dataset, although further error analysis is needed to understand its limitations and reliability on unseen real-world news.

### Error Analysis

The aim for this is to investigate the articles the linear svm classified incorrectly and try to undestand why.

In [68]:
# create a dataframe of the predictions
error_df = pd.DataFrame({
    'article': X_test,
    'actual_label': y_test,
    'predicted_label': y_pred_svm
})

# filter the table to keep only incorrect predictions
error_df = error_df[
    error_df['actual_label'] != error_df['predicted_label']
].copy()

# show samples of the mistakes for investigation
print("Total misclassified articles:", len(error_df))
print(error_df.head())

Total misclassified articles: 80
                                                article actual_label  \
3437  hillary clinton enjoys solid lead in early vot...         FAKE   
1523  fear of trump sparking a surge in citizenship,...         REAL   
7317  memo to trump: 'action this day!' => "in victo...         FAKE   
6859  kate steinle's death and sanctuary cities: wha...         REAL   
2153  clinton emails on trade deal held until after ...         REAL   

     predicted_label  
3437            REAL  
1523            FAKE  
7317            REAL  
6859            FAKE  
2153            FAKE  


In [69]:
# Check the mistakes


print(
    pd.crosstab(
        error_df['actual_label'],
        error_df['predicted_label'],
        rownames=['Actual'],
        colnames=['Predicted']
    )
)

Predicted  FAKE  REAL
Actual               
FAKE          0    30
REAL         50     0


These are how many REAL articles were classified as FAKE and how many FAKE articles were classified as REAL.

In [70]:
# Inspect some of the misclassified Artices

for _, row in error_df.head(5).iterrows():
    print("Actual label:   ", row['actual_label'])
    print("Predicted label:", row['predicted_label'])
    print("Article:")
    print(row['article'][:1000])
    print("-" * 80)

Actual label:    FAKE
Predicted label: REAL
Article:
hillary clinton enjoys solid lead in early voting: reuters/ipsos poll by reuters 5:51 pm democratic nominee hillary clinton leads republican donald trump by 15 percentage points among early voters surveyed in the past two weeks, according to the reuters/ipsos states of the nation project. by maurice tamman new york (reuters) – with 11 days to go before the u.s. presidential election, democratic nominee hillary clinton leads republican donald trump by 15 percentage points among early voters surveyed in the past two weeks, according to the reuters/ipsos states of the nation project. though data is not available for all early voting states, clinton enjoys an edge in swing states such as ohio and arizona and in republican party strongholds such as georgia and texas. an estimated 19 million americans have voted so far in the election, according to the university of florida's united states election project, accounting for as much as 20 per

Three useful observations i found out:
- Text completeness matters. Short headlines or incomplete articles provide less information than full articles.
- Writing style can be misleading. News reporting, commentary, and emotionally worded text may overlap across the two labels.
- The dataset labels need context. An incorrect prediction relative to the dataset does not automatically mean the article's label is objectively correct or incorrect.

Next: Inspect the model's remaining errors to see whether these patterns are widespread or whether the five examples are unusual.

In [71]:
# Count errors by actual and predicted class
print(
    error_df.groupby(
        ['actual_label', 'predicted_label']
    ).size()
)

# Inspect the shortest misclassified articles
error_df['article_length'] = error_df['article'].str.len()

print(
    error_df[
        ['actual_label', 'predicted_label', 'article_length', 'article']
    ]
    .sort_values('article_length')
    .head(10)
    .to_string(index=False)
)

actual_label  predicted_label
FAKE          REAL               30
REAL          FAKE               50
dtype: int64
actual_label predicted_label  article_length                                                                                                                                                                                                                                                                                                                                                                                                                                                                           article
        REAL            FAKE              82                                                                                                                                                                                                                                                                                                                                          

With these output: I have analyzed the Articles text with LLM, and there are two recurring issues:

- Short or incomplete text: Some inputs combine a headline with unrelated text, end abruptly, or contain very little context.
- Mixed content: Some articles appear to contain snippets from multiple stories. For example, a headline may be followed by text about a different person or topic.


These patterns suggest possible data-quality problems, it is not necessarily flaws in the SVM algorithm. The model classifies the text it receives; and it cannot reliably reconstruct missing context. 

- So far this error Analysis has provided a good pattern recognition on why some misclassification occurs.

## Save Model

I will save the Linear SVM and TF-IDF vectorizer as a single scikit-learn pipeline. It will make this Fake News Detection project easier to reuse and deploy.

This model is selected because it outperformed the logistic Regression.

In [73]:
# import pipeline and preprocessing function

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

In [75]:
# Define a preprocessing function to process thye text into a more consistent and readable format

import re

def preprocess_texts(texts):
    cleaned_texts = []

    for text in texts:
        text = str(text).lower()
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'<.*?>', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        cleaned_texts.append(text)

    return cleaned_texts

In [76]:
fake_news_pipeline = Pipeline([
    # Text preprocessing
    ('preprocessing', FunctionTransformer(
        preprocess_texts,
        validate=False
    )),
    # the tfidf converts raw text into numerical features.
    ('tfidf', TfidfVectorizer(  
        max_features=50000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95
    )),
    # the svm classifies the features as FAKE or REAL.
    ('svm', LinearSVC(
        random_state=42,
        max_iter=2000
    ))
])

In [77]:
# prepare the raw data and split with the already concatenated text and title column (model_text)
X = df_clean['model_text']
y = df_clean['label']

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train_raw))
print("Testing samples:", len(X_test_raw))

Training samples: 4831
Testing samples: 1208


In [78]:
# pipeline training with the training preprocessed data 
# This will fit the TF-IDF vectorizer on the training text and then trains Linear SVM on the resulting features.

fake_news_pipeline.fit(X_train_raw, y_train_raw)
print("Pipeline training completed")

Pipeline training completed


In [79]:
# Evaluate the pipeline to ensure it performs well as expected on the test data

y_pred_pipeline = fake_news_pipeline.predict(X_test_raw)

print("Pipeline Accuracy:", round(
    accuracy_score(y_test_raw, y_pred_pipeline), 4
))

print(classification_report(y_test_raw, y_pred_pipeline))

Pipeline Accuracy: 0.9338
              precision    recall  f1-score   support

        FAKE       0.92      0.95      0.94       612
        REAL       0.95      0.92      0.93       596

    accuracy                           0.93      1208
   macro avg       0.93      0.93      0.93      1208
weighted avg       0.93      0.93      0.93      1208



In [80]:
# Save the model using joblib
import joblib

joblib.dump(fake_news_pipeline, 'fake_news_pipeline.joblib')

# confirm the saving is successfull
print("Pipeline saved successfully")



Pipeline saved successfully


In [81]:
# Load and verify the loaded pipeline

loaded_pipeline = joblib.load(
    'fake_news_pipeline.joblib'
)

print("Pipeline loaded successfully.")

Pipeline loaded successfully.


In [84]:
# compare predictions from the original fitted pipeline and the loaded one

sample_article = X_test_raw.iloc[0]

original_predictions = fake_news_pipeline.predict(sample_article)
loaded_predictions = loaded_pipeline.predict([sample_article])

# Displays the first predictions and assertain if the two predictions are matched
print("Original predictions:", original_predictions)
print("Loaded predictions:  ", loaded_predictions)

assert (original_predictions == loaded_predictions).all()

print("Verification successful: predictions match.")

Original predictions: ['FAKE' 'FAKE' 'FAKE' ... 'FAKE' 'FAKE' 'FAKE']
Loaded predictions:   ['FAKE']
Verification successful: predictions match.


 #### Test a new article

Test the model with a new input that wasn't taken from the test set 

In [85]:
new_article = """
Example headline: Local authorities announce a new public transport plan.
The transport department released details of the proposal on Monday.
"""

prediction = loaded_pipeline.predict([new_article])

print("Predicted label:", prediction[0])

Predicted label: REAL
